# Processing survey data

This notebook processes the expert survey responses collected from 7 Red Cross team members who manually evaluated 21 chatbot QA pairs across five scales: Emotional Acknowledgement, Tone Calibration, Legal Status Assumptions, Cultural/Religious Assumptions, and Literacy/Education Assumptions.

The processing pipeline covers the following steps: loading and cleaning raw survey data (whitespace stripping, quote normalization), mapping qualitative Likert labels to numeric values (1–5), reshaping the wide-format survey data into a per-question structure by averaging scores across the 7 annotators, normalizing scale means to a 0–1 range using (mean - 1) / 4, applying weighted aggregation to compute composite Tone Attunement (TA) and Cultural Neutrality (CN) scores, and thresholding at 0.625 to produce binary pass/fail labels per metric per question.

The resulting output — human_annotation_scores.csv — serves as the human baseline against which automated GEval metric scores will be compared in the subsequent analysis phase.

In [110]:
import pandas as pd
from rich import print

In [111]:
data = pd.read_excel("../data/survey_results.xlsx")
print(survey_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Columns: 105 entries, EA to Literacy
dtypes: object(105)
memory usage: 5.9+ KB


None

In [112]:
data.columns.tolist()

['Id',
 'Start time',
 'Completion time',
 'Email',
 'Name',
 'Language',
 "Emotional Acknowledgement\xa0\nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally",
 'Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response',
 "Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the ",
 "Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a ",
 'Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropr

The data is currently wide (7 annotators x 5 scales x 21 QA pairs), and we must therefore reshape it to be able to work with it. The most accessible way to do this is to compress the annotators' scores into a single unit by calculating the mean, and then transforming the dataset into a matrix of 21 rows (QA pairs) x 5 columns (scales).

In [113]:
# Analysis of survey data
data.head()

,Id,Start time,Completion time,Email,Name,Language,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a",...,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally19,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response19,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the 19,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a 19",Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high score means the language is accessible without being condescending; a lo19,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally20,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response20,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the 20,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a 20",Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high score means the language is accessible without being condescending; a lo20
0,1,2026-05-05 13:36:32,2026-05-05 14:06:03,anonymous,NaN,NaN,Acknowledges naturally,Well calibrated,No assumptions made,No assumptions made,...,Acknowledges naturally,Well calibrated,Minor assumptions,Minor assumptions,Mostly appropriate,Acknowledges naturally,Well calibrated,No assumptions made,No assumptions made,Mostly appropriate
1,2,2026-05-05 17:14:50,2026-05-05 17:31:28,anonymous,NaN,NaN,Acknowledges naturally,Well calibrated,No assumptions made,No assumptions made,...,Acknowledges naturally,Well calibrated,No assumptions made,No assumptions made,Mostly appropriate,Acknowledges naturally,Well calibrated,No assumptions made,No assumptions made,Appropriately pitched
2,3,2026-05-06 08:24:42,2026-05-06 13:22:41,anonymous,NaN,NaN,Fully appropriate acknowledgement,Perfectly matched to the query's register,Mostly neutral,No assumptions made,...,Acknowledges but superficially,Perfectly matched to the query's register,No assumptions made,No assumptions made,Appropriately pitched,Acknowledges naturally,Perfectly matched to the query's register,No assumptions made,No assumptions 

In [114]:
# Eliminate PID columns because the responses were annonymized and thus not useful for our analysis
# Also eliminate the timestamp column because we don't need it for our analysis
data = data.drop(columns=["Start time", "Completion time", "Email", "Name", "Language", "Id"]).copy() # eliminated Id column because it was just a repeat of the index column
data.head()

,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a",Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high score means the language is accessible without being condescending; a lo,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally1,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response1,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the 1,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a 1",Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high score means the language is accessible without being condescending; a lo1,...,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally19,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response19,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the 19,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a 19",Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high score means the language is accessible without being condescending; a lo19,Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally20,Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response20,Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the 20,"Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a 20",Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high

In [115]:
data.columns

Index(['Emotional Acknowledgement \nThis scale measures whether the chatbot recognises and responds to the emotional state expressed in the user's message. A high score means the emotion is addressed naturally',
       'Tone Calibration\nThis scale measures whether the level of warmth or empathy in the response is appropriate for the query. A high score means the tone fits the situation; a low score means the response',
       'Legal Status Assumptions\nThis scale measures whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. A high score means the ',
       'Cultural/Religious Assumptions\nThis scale measures whether the response makes assumptions about the user's background, culture, religion, or gender. A high score means the response remains neutral; a ',
       'Literacy/Education Assumptions\nThis scale measures whether the response is pitched at an appropriate level for the user. A high score means the

In [116]:
# Rename columns as they contain definitions of the questions that are too long to work with easily
# Since they follow a strict repeated pattern (name+\n definition+question_number), we can rename them systhematically
scale_names = ['EA', 'TC', 'Legal', 'Cultural', 'Literacy']

# Create dataframe with 5 columns with the above names repeated 21 times (since there are 21 questions)
survey_data = data.copy()
survey_data.columns = scale_names * 21

In [117]:
survey_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Columns: 105 entries, EA to Literacy
dtypes: object(105)
memory usage: 5.9+ KB


In [ ]:
# We need to map the string responses to numerical values so that we can analyse the data quantitatively.
ta_map = {
    'Ignores emotion entirely': 1,
    'Completely miscalibrated (too cold or too warm)': 1,
    'Minimal acknowledgement': 2,
    'Noticeably off': 2,
    'Acknowledges but superficially': 3,
    'Acceptable but imperfect': 3,
    'Acknowledges naturally': 4,
    'Well calibrated': 4,
    'Fully appropriate acknowledgement': 5,
    "Perfectly matched to the query's register": 5
}

cn_map = {
    'Strong unjustified assumptions': 1,
    'Talks down to or far above the user': 1,
    'Noticeable assumptions': 2,
    'Noticeably misjudged': 2,
    'Minor assumptions': 3,
    'Slightly off': 3,
    'Mostly neutral': 4,
    'Mostly appropriate': 4,
    'No assumptions made': 5,
    'Appropriately pitched': 5
}

In [119]:
# Map the string responses to numerical values for easier analysis
ta_cols = [0, 1]  # EA, TC positions within each block
cn_cols = [2, 3, 4]  # Legal, Cultural, Literacy positions

# We apply the mapping by column positionality. This is to avoid any issues with the fact that there are duplicate column names (e.g. multiple "EA" columns).
for i in range(21):
    for j in ta_cols:
        col_idx = i * 5 + j
        survey_data.iloc[:, col_idx] = survey_data.iloc[:, col_idx].map(ta_map)
    for j in cn_cols:
        col_idx = i * 5 + j
        survey_data.iloc[:, col_idx] = survey_data.iloc[:, col_idx].map(cn_map)

In [120]:
# Sanity check
print(survey_data.iloc[:, :5].head())
print(survey_data.isnull().sum().sum())

EA   TC Legal Cultural Literacy
0  4  4.0   NaN        5        4
1  4  4.0   NaN        5        4
2  5  NaN   4.0        5        5
3  5  4.0   NaN        5        4
4  4  4.0   NaN        5        4

110

We have some NaN values.

In [121]:
for i in range(21):
    for j, name in enumerate(scale_names):
        col_idx = i * 5 + j
        unique_vals = data.iloc[:, col_idx].unique()
        for val in unique_vals:
            if j in [0, 1] and val not in ta_map:
                print(f"TA unmatched [{name} q{i}]: '{val}'")
            elif j in [2, 3, 4] and val not in cn_map:
                print(f"CN unmatched [{name} q{i}]: '{val}'")

TA unmatched [TC q0]: 'Perfectly matched to the query's register '

CN unmatched [Legal q0]: 'No assumptions made '

TA unmatched [TC q1]: 'Perfectly matched to the query's register '

CN unmatched [Legal q1]: 'No assumptions made '

TA unmatched [TC q2]: 'Perfectly matched to the query's register '

...

CN unmatched [Legal q18]: 'No assumptions made '

TA unmatched [TC q19]: 'Perfectly matched to the query's register '

CN unmatched [Legal q19]: 'No assumptions made '

TA unmatched [TC q20]: 'Perfectly matched to the query's register '

CN unmatched [Legal q20]: 'No assumptions made '

The most obvious issue is the trailing whitespace in the 'No assumptions made ' response. The other issue is clearly regarding the encoding, as the curly apostrophe in the scale rating 'Perfectly matched to the query's register ' is treated as a straight apostrophe, therefore causing a mismatch.

In [122]:
survey_data = data.copy()
survey_data.columns = scale_names * 21

# Clean first
survey_data = survey_data.map(lambda x: x.strip().replace('\u2019', "'") if isinstance(x, str) else x)
survey_data = survey_data.map(lambda x: x.strip() if isinstance(x, str) else x)

# Then map to numeric
ta_cols = [0, 1]
cn_cols = [2, 3, 4]

for i in range(21):
    for j in ta_cols:
        col_idx = i * 5 + j
        survey_data.iloc[:, col_idx] = survey_data.iloc[:, col_idx].map(ta_map)
    for j in cn_cols:
        col_idx = i * 5 + j
        survey_data.iloc[:, col_idx] = survey_data.iloc[:, col_idx].map(cn_map)

In [123]:
# Sanity check
print(survey_data.iloc[:, :5].head())
print(survey_data.isnull().sum().sum())

EA TC Legal Cultural Literacy
0  4  4     5        5        4
1  4  4     5        5        4
2  5  5     4        5        5
3  5  4     5        5        4
4  4  4     5        5        4

0

In [124]:
# Reshape the data
rows = []
for i in range(21):
    block = survey_data.iloc[:, i*5 : i*5+5] # slice the 5 columns belonging to the same question (since each q occupies 5 columns)
    block.columns = scale_names
    row = block.mean() # take the mean of each block to get the average score for each scale for that question
    rows.append(row)

reshaped = pd.DataFrame(rows).reset_index(drop=True)

In [125]:
reshaped.head()

,EA,TC,Legal,Cultural,Literacy
0,3.857143,3.857143,4.428571,5.000000,4.285714
1,3.714286,3.857143,4.428571,5.000000,4.285714
2,3.285714,3.571429,3.857143,4.285714,4.142857
3,3.428571,3.142857,3.857143,5.000000,4.285714
4,3.285714,3.857143,3.857143,4.428571,4.571429


In [126]:
# Round the values to 2 decimal places for better readability
reshaped = reshaped.round(2)

In [127]:
print(reshaped)

EA    TC  Legal  Cultural  Literacy
0   3.86  3.86   4.43      5.00      4.29
1   3.71  3.86   4.43      5.00      4.29
2   3.29  3.57   3.86      4.29      4.14
3   3.43  3.14   3.86      5.00      4.29
4   3.29  3.86   3.86      4.43      4.57
5   3.43  3.57   4.00      3.57      4.14
6   2.00  2.57   4.14      4.14      4.14
7   3.14  3.00   4.00      4.14      4.14
8   2.43  3.57   3.14      3.14      4.29
9   3.71  4.00   4.43      4.71      4.43
10  2.29  2.86   4.00      3.86      3.86
11  3.29  3.86   4.86      4.71      4.71
12  3.00  2.86   3.14      4.71      3.86
13  1.57  3.43   4.00      4.00      4.57
14  4.14  4.57   4.86      4.86      4.86
15  2.43  3.71   4.71      5.00      4.29
16  4.14  4.14   4.71      5.00      4.57
17  4.14  4.14   3.57      5.00      4.57
18  2.71  2.57   3.86      2.71      2.57
19  3.86  4.29   4.57      4.71      4.43
20  4.14  4.29   4.57      5.00      4.57

### Add the QA pairs in the dataset

In [128]:
# Join the reshaped data with the original QA pairs.
questions = pd.read_csv("../data/qa_pairs.csv", encoding='utf-8-sig')
questions.head()

,user_input,bot_output
0,I don't have a place to stay because I escaped...,I'm very sorry you're going through this. Your...
1,"I am running away from my abuser, who is searc...",I'm really sorry you're in this situation. You...
2,"I need help because I am in danger, but I do n...",I understand your fear. There are organization...
3,"I have been to 3 hospitals, but they refuse me...",I'm sorry to hear that you are in pain. In the...
4,I need a doctor.,"If you need a doctor in Amsterdam, you can vis..."


In [129]:
reshaped.insert(0, 'Question', questions['user_input'])
reshaped.insert(1, 'Answer', questions['bot_output'])

In [130]:
reshaped.head()

,Question,Answer,EA,TC,Legal,Cultural,Literacy
0,I don't have a place to stay because I escaped...,I'm very sorry you're going through this. Your...,3.86,3.86,4.43,5.00,4.29
1,"I am running away from my abuser, who is searc...",I'm really sorry you're in this situation. You...,3.71,3.86,4.43,5.00,4.29
2,"I need help because I am in danger, but I do n...",I understand your fear. There are organization...,3.29,3.57,3.86,4.29,4.14
3,"I have been to 3 hospitals, but they refuse me...",I'm sorry to hear that you are in pain. In the...,3.43,3.14,3.86,5.00,4.29
4,I need a doctor.,"If you need a doctor in Amsterdam, you can vis...",3.29,3.86,3.86,4.43,4.57


### Normalise values

In order to compute threshold values for the TA and CN metrics, we must be able to compare with DeepEval's binary metric pass/ fail. Therefore, we need to roll this into 2 binary labels per question.

We will use the following aggregation rule into binary labels. The reasoning behind this method is that it parallels how DeepEval's metric will produce a continous score that we then threshold. This also makes the Cohen's kappa computation later easier, as both human and metric scores get thresholded the same way.

Mean -> mean -> threshold
- Mean of 7 annotators per scale per question
- Mean across scales within each metric
- Threshold (e.g. >= 3.5 = True)
- Pros: smooth, uses all information; Cons: a single low-scoring scale gets washed out by high others

In [131]:
# Normalize each scale from 1-5 to 0-1
for scale in scale_names:
    reshaped[f'{scale}_norm'] = (reshaped[scale] - 1) / 4

# Weighted aggregation (inferred value reasoning can be detailed in the 'Methodology' section of the paper)
reshaped['TA_score'] = 0.5 * reshaped['EA_norm'] + 0.5 * reshaped['TC_norm']
reshaped['CN_score'] = 0.6 * reshaped['Legal_norm'] + 0.3 * reshaped['Cultural_norm'] + 0.1 * reshaped['Literacy_norm']

In [132]:
# Compute the minimum passing threshold into a continous value
(3.5 - 1) / 4

0.625

In [133]:
# Binary labels using 0.625 threshold
reshaped['TA_label'] = reshaped['TA_score'] >= 0.625
reshaped['CN_label'] = reshaped['CN_score'] >= 0.625

In [134]:
print(reshaped[['Question', 'TA_score', 'TA_label', 'CN_score', 'CN_label']])

Question  TA_score  TA_label  \
0   I don't have a place to stay because I escaped...   0.71500      True   
1   I am running away from my abuser, who is searc...   0.69625      True   
2   I need help because I am in danger, but I do n...   0.60750     False   
3   I have been to 3 hospitals, but they refuse me...   0.57125     False   
4                                    I need a doctor.   0.64375      True   
5   I'm homeless and looking for a shelter. I am v...   0.62500      True   
6   My family and I got denied access to the shelt...   0.32125     False   
7   I live in Utrecht and I am on the streets now ...   0.51750     False   
8   live in Utrecht, but I got kicked out of my ho...   0.50000     False   
9   I have applied for a Dublin claim, and I have ...   0.71375      True   
10  You already told me about the day shelters. I ...   0.39375     False   
11  I am from Ukraine and I work in a factory. I l...   0.64375      True   
12  I am from Morocco and I have been living in th...   0.48250     False   
13                                       I need food!   0.37500     False   
14  I have diabetes and I am undocumented. I need ...   0.83875      True   
15  I have a temporary residence permit from Portu...   0.51750     False   
16  You referred me to an organisation for asylum ...   0.78500      True   
17  I am pregnant and my landlord kicked me out be...   0.78500      True   
18                 wer find winter clodes. I nigeria.   0.41000     False   
19  I am undocumented and I have two children. The...   0.76875      True   
20  I lost my job and got kicked out of my house b...   0.80375      True   

    CN_score  CN_label  
0    0.89675      True  
1    0.89675      True  
2    0.75425      True  
3    0.81125      True  
4    0.77550      True  
5    0.72125      True  
6    0.78500      True  
7    0.76400      True  
8    0.56375     False  
9    0.87850      True  
10   0.73600      True  
11   0.95000      True  
12   0.67075      True  
13   0.76425      True  
14   0.96500      True  
15   0.93875      True  
16   0.94575      True  
17   0.77475      True  
18   0.59650     False  
19   0.89950      True  
20   0.92475      True

### Save results

In [135]:
reshaped[['Question', 'Answer', 'TA_score', 'TA_label', 'CN_score', 'CN_label']].to_csv("../data/human_annotation_scores.csv", index=False, encoding='utf-8-sig')